<a href="https://www.kaggle.com/code/maria561/china-cleantech-exports?scriptVersionId=365591279" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:

%%capture
!pip install datasist scikit-learn category-encoders joblib streamlit pandas numpy

In [3]:
import pandas as pd 
import numpy as np
import plotly.express as px
from datasist.structdata import detect_outliers

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler ,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor


from sklearn.model_selection import cross_validate,TimeSeriesSplit,GridSearchCV
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor,ColumnTransformer
from category_encoders import BinaryEncoder
import joblib


# Data Cleaning and Feature Engineering

In [4]:
df = pd.read_csv('/kaggle/input/datasets/maria561/china-cleantech-exports-energy-ember/clean_tech_exports_full_release_monthly.csv',
                 index_col=False)

In [ ]:
df.info()

In [8]:
# checking categorical features variance

for col in df[['Area','Area type','Region','Commodity category']]:
    print(f"no of unique: {df[col].nunique()}")
    print(df[col].value_counts().sort_values(ascending=True))
    print('*'*50)

no of unique: 250
Area
Pitcairn Islands                                     2
Saint Helena and Ascension and Tristan da Cunha      4
Falkland Islands (Malvinas)                          7
Norfolk Island                                       7
Niue                                                 9
                                                  ... 
Slovenia                                           735
Singapore                                          735
Sierra Leone                                       735
Serbia                                             735
Senegal                                            735
Name: count, Length: 250, dtype: int64
**************************************************
no of unique: 2
Area type
Region                  9555
Country or economy    150626
Name: count, dtype: int64
**************************************************
no of unique: 7
Region
North America                   1565
Middle East                    10290
Oceania                 

In [5]:
# dropping Area Type feature because of very low variance 

df = df.drop(['Area type'],axis=1)

In [ ]:
# ckecking outliers in numerical features

for col in df.select_dtypes(include='float64'):
   outliers =detect_outliers(df,features=[col],n=0)
   print(f'{col} outliers% = ',(len(outliers)/len(df[col]))*100)

In [6]:
# extracting year and month from Date feature

df['Date'] = pd.to_datetime(df['Date'],format='%Y-%m-%d')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

In [7]:
df = df.drop(['Date'],axis=1)

In [9]:
# checking duplicates

print(df.duplicated().sum())


0


In [8]:
# making a copy of the dataframe for deployment and EDA

df_eda = df.copy()

In [9]:
# filling missing values

df_eda['Region'] = df_eda['Region'].fillna('Unknown')

# filling by median because of outliers
df_eda['Amount 12 month rolling sum (USD)'] = df_eda['Amount 12 month rolling sum (USD)'].fillna(df_eda['Amount 12 month rolling sum (USD)'].median())

In [13]:
# checking duplicates after modifying 

df_eda.duplicated().sum()

np.int64(0)

In [16]:
df_eda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160181 entries, 0 to 160180
Data columns (total 7 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Area                               160181 non-null  object 
 1   Region                             160181 non-null  object 
 2   Commodity category                 160181 non-null  object 
 3   Amount (USD)                       160181 non-null  float64
 4   Amount 12 month rolling sum (USD)  160181 non-null  float64
 5   Year                               160181 non-null  int32  
 6   Month                              160181 non-null  int32  
dtypes: float64(2), int32(2), object(3)
memory usage: 7.3+ MB


In [15]:
# summary statistics 

df_eda.describe(include = 'number')

,Amount (USD),Amount 12 month rolling sum (USD),Year,Month
count,1.601810e+05,1.601810e+05,160181.000000,160181.000000
mean,7.707255e+07,9.195227e+08,2021.955994,6.391039
std,5.852700e+08,6.251196e+09,2.531972,3.406472
min,0.000000e+00,0.000000e+00,2018.000000,1.000000
25%,7.817000e+03,1.036136e+06,2020.000000,3.000000
50%,4.943070e+05,1.650162e+07,2022.000000,6.000000
75%,6.696533e+06,3.653428e+08,2024.000000,9.000000
max,3.184280e+10,2.902016e+11,2026.000000,12.000000


In [10]:
# exporting clean dataset

df_eda.to_csv('china_exports_cleaned.csv',index=False)

In [12]:
import os

output_path = "/kaggle/working/china_exports_cleaned.csv"

df_eda.to_csv(output_path, index=False)


File exists: True
File size: 9361173


# EDA

Univariate Analysis

In [ ]:
for col in df_eda.select_dtypes(include='float64').columns:
    distribution = px.histogram(df_eda, x=col, title=f'Distribution of {col}',nbins=5,marginal='rug',
                                color_discrete_sequence=['cornflowerblue'])
    distribution.show()

In [ ]:
# visualizing outliers

for col in df_eda[['Amount (USD)','Amount 12 month rolling sum (USD)']]:
    box = px.box(df_eda, y = col,color_discrete_sequence=['dodgerblue'])
    box.show()

Bivariate Analysis

In [ ]:
# 'Amount (USD)' vs. 'Commodity category'


# excluding 'All technologies'
category_totals = (df_eda[df_eda['Commodity category'] != 'All technologies']
    .groupby('Commodity category')['Amount (USD)'].sum().reset_index().sort_values('Amount (USD)', ascending=False))


bar_1= px.bar( category_totals, x='Commodity category',y='Amount (USD)',
    title='Exports by Commodity Category',color_discrete_sequence=['skyblue'])

bar_1.show()

In [ ]:
# 'Amount (USD)' vs. 'Region'


# excluding "unknown" from regions 
region_totals = (df_eda[df_eda['Region']!='Unknown'].groupby('Region')['Amount (USD)'].sum().reset_index().sort_values('Amount (USD)', ascending=False))


bar_2= px.bar(region_totals,x='Region', y='Amount (USD)',
    title='Exports by Region',color_discrete_sequence=['deepskyblue'])

bar_2.show()

In [ ]:
#Time Series Analysis

# Average exports trend over years

mean = df_eda.groupby('Year')['Amount (USD)'].mean().reset_index()
line = px.line(mean,y='Amount (USD)',x='Year',title='Exports Trend Over Time',color_discrete_sequence=['dodgerblue'])
line.show()


Seasonality Analysis

In [ ]:
# exploring seasonality patterns by category


# excluding "All technologies" from categories feature
categories = ['Batteries','EVs','Grid','Heating and cooling systems','Solar PV', 'Wind']

for category in categories:

    commodity = d_eda[df_eda['Commodity category'] != 'All technologies']

    total_commodity = ( commodity.groupby('Month')['Amount (USD)'].sum().reset_index())

    line_2 = px.line( monthly_avg, x='Month',color_discrete_sequence=['deeppink'],
        y='Amount (USD)', markers=True, title=f'Monthly Seasonality - {category}')

    line_2.show()

# Machine Learning 

Data Preprocessing

In [15]:
# removing null values from target variable

df = df.dropna(subset ='Amount 12 month rolling sum (USD)' )

In [16]:
y = df['Amount 12 month rolling sum (USD)']
x = df.drop(columns = 'Amount 12 month rolling sum (USD)')

In [17]:
numerical_features = ['Amount (USD)','Year','Month']
binary_categorical = ['Area']
one_hot_categorical = ['Commodity category','Region']

In [18]:
numerical_pipeline = Pipeline([('scaling',RobustScaler())])
one_hot_pipeline = Pipeline([('onehot',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'))])

binary_pipeline = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                            ('binary',BinaryEncoder())])

In [30]:
preprocessing = ColumnTransformer(transformers=[('numerical',numerical_pipeline,numerical_features),
                                                ('one_hot',one_hot_pipeline,one_hot_categorical),
                                                ('binary',binary_pipeline,binary_categorical)],
                                                remainder='passthrough')
preprocessing

ColumnTransformer(remainder='passthrough',
                  transformers=[('numerical',
                                 Pipeline(steps=[('scaling', RobustScaler())]),
                                 ['Amount (USD)', 'Year', 'Month']),
                                ('one_hot',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Commodity category', 'Region']),
                                ('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('binary', BinaryEncoder())]),
                                 ['Area'])])

Baseline Modelling and Cross Validation

In [53]:
model_base = [('KNN',KNeighborsRegressor()),
              ('Desicion Tree',DecisionTreeRegressor(random_state=42)),
              ('Random Forest',RandomForestRegressor(random_state=42)),
              ('XGB', XGBRegressor(random_state=42))]

In [ ]:
# time_based cross validation

df_cv = df.copy()

df_cv['Date'] = pd.to_datetime(dict(year=df['Year'], month=df['Month'], day=1))

df_cv = df_cv.sort_values('Date').reset_index(drop=True)


#  unique months in chronological order
unique_dates = np.sort(df_cv['Date'].unique())

# Create 5 chronological folds
ts_cv = TimeSeriesSplit(n_splits=5)

# Convert month splits into row index splits for equivalent train/validation splits with respect to month
cv_splits = []

for train_month_idx, val_month_idx in ts_cv.split(unique_dates):

    train_dates = unique_dates[train_month_idx]
    val_dates = unique_dates[val_month_idx]

    # return only if True then convert into numpy array
    train_rows = np.flatnonzero(df_cv['Date'].isin(train_dates).to_numpy())

    val_rows = np.flatnonzero(df_cv['Date'].isin(val_dates).to_numpy())

    cv_splits.append((train_rows, val_rows))



In [54]:
for name,model in model_base:
    
    model_pipeline=Pipeline(steps=[('preprocessing',preprocessing),('model',model)])

    model_pipeline_scaled=TransformedTargetRegressor(regressor=model_pipeline,func=np.log1p,inverse_func=np.expm1)

    result=cross_validate(model_pipeline_scaled,x,y, cv=cv_splits,scoring='r2',return_train_score=True,n_jobs=1)

    print(name)
    print('train_score',result['train_score'].mean())
    print('test_score',result['test_score'].mean())
    print('*'*50)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


KNN
train_score 0.9379967825450969
test_score 0.7729844609317692
**************************************************


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Desicion Tree
train_score 1.0
test_score 0.6570433386305696
**************************************************


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Random Forest
train_score 0.9922011567712848
test_score 0.7647936046660149
**************************************************


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


XGB
train_score 0.9551874401405819
test_score 0.3719449730963043
**************************************************


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


KNN model is the top performer on this data in test scores

# Hyperparameter Tuning

In [65]:
parameter_grid =  { 'KNN__n_neighbors': [3, 5, 7, 9],
                    'KNN__weights': ['uniform', 'distance'],
                    'KNN__p': [1, 2]}


knn_pipeline = Pipeline(steps=[('preprocesing',preprocessing),
                                  ('KNN',KNeighborsRegressor())])

In [68]:
grid_result = GridSearchCV(knn_pipeline,parameter_grid,cv=4,scoring='r2',
                           return_train_score=True,n_jobs=-1)

grid_result.fit(x, y)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

GridSearchCV(cv=4,
             estimator=Pipeline(steps=[('preprocesing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numerical',
                                                                         Pipeline(steps=[('scaling',
                                                                                          RobustScaler())]),
                                                                         ['Amount '
                                                                          '(USD)',
                                                                          'Year',
                                                                          'Month']),
                                                                        ('one_hot',
                                                                         Pipeline(steps=[('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='ignore',
                                                                                                        sparse_output=False))]),
                                                                         ['Commodity '
                                                                          'category',
                                                                          'Region']),
                                                                        ('binary',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('binary',
                                                                                          BinaryEncoder())]),
                                                                         ['Area'])])),
                                       ('KNN', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'KNN__n_neighbors': [3, 5, 7, 9], 'KNN__p': [1, 2],
                         'KNN__weights': ['uniform', 'distance']},
             return_train_score=True, scoring='r2')

In [69]:
print(grid_result.best_score_)
print(grid_result.best_params_)

0.7551803458659166
{'KNN__n_neighbors': 9, 'KNN__p': 1, 'KNN__weights': 'distance'}


In [71]:
tuned_pipeline = Pipeline(steps=[('preprocessing',preprocessing),
                               ('KNN',KNeighborsRegressor(n_neighbors=9 , p=1 ,
                                            weights='distance', n_jobs=-1))])
tuned_pipeline.fit(x,y)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('scaling',
                                                                   RobustScaler())]),
                                                  ['Amount (USD)', 'Year',
                                                   'Month']),
                                                 ('one_hot',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Commodity category',
                                                   'Region']),
                                                 ('binary',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('binary',
                                                                   BinaryEncoder())]),
                                                  ['Area'])])),
                ('KNN',
                 KNeighborsRegressor(n_jobs=-1, n_neighbors=9, p=1,
                                     weights='distance'))])

In [72]:
joblib.dump(tuned_pipeline,'knn.pkl',compress=3)

['knn.pkl']

In [16]:
%%writefile /kaggle/working/china_tech.py



import pandas as pd
import streamlit as st
from streamlit_extras.metric_cards import style_metric_cards
import plotly.express as px
import joblib
import category_encoders 
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from category_encoders import BinaryEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer



# setting up streamlit dashboard and loading the data and the model 

st.set_page_config(layout='wide',page_title='Power Plants')
st.title('China Cleantech Exports - Energy Ember')

df=pd.read_csv('china_exports_cleaned.csv')

knn = joblib.load('knn.pkl')


# Dividing dashboard into multiple sections


# overview page

page=st.sidebar.radio('select page',['Overview','Analysis','Forecasting'])


if page == 'Overview':
    st.dataframe(df)

    # explaining features meaning
    cols = {'Area':'Country or geographical area associated with the recorded data.',
            'Region	':'Geographical region to which the area belongs.',
            'Commodity category':'Clean technology category represented in the data.',
            'Amount (USD)':'Recorded monetary value for the commodity category in USD for the given month.',
            'Amount 12 month rolling sum (USD)':'Total monetary value over the preceding 12 months, including the current month.',
            'Year'	:'Calendar year of the observation.',
            'Month' :'month of the observation' }
                    
    for col, meaning in cols.items():
        with st.sidebar.expander(col):
            st.success(meaning)





    def metrics() :
        col1, col2, col3 , col4 = st.columns(5)

        total_countries = len(df['Area'].unique())
        avg_amount = round(df['Amount (USD)'].mean(),2)
        min_amount= round(df['Amount (USD)'].min(),2)
        max_amount= round(df['Amount (USD)'].max(),2)



        col1.metric(label='Total Countries', value=total_countries)
        col2.metric(label="Total Categories", value= 6)
        col3.metric(label='Average Amount', value=avg_amount)
        col4.metric(label='Min Amount', value=min_amount)
        col5.metric(label='Max Amount', value=max_amount)

        style_metric_cards()

    
       # analysis and visualization page

elif page == 'Analysis' :
    tab1,tab2,tab3,tab4,tab5 = st.tabs(['Distributions','Exports by  Category',
                                             'Exports by Region','Time Series - Amount', 
                                             'Time Series - Seasonality by Category')
    
    with tab1 :
        col6,col7= st.columns(2,gap = 'large')
       

        with col6 :

            fig_1 = px.histogram(df, x=col, title='Distribution of Amouny (USD)',nbins=5,marginal='rug',
                                color_discrete_sequence=['cornflowerblue'])
            fig_1.show()
            st.plotly_chart(fig1,use_container_width=True)

        with col7 :
            
                fig_2 = px.histogram(df, x=col, title='Distribution of Rolling Sum (USD)',nbins=5,marginal='rug',
                                color_discrete_sequence=['cornflowerblue'])
                fig_2.show()
                st.plotly_chart(fig1,use_container_width=True)

    with tab2:
        category_totals = (df[df['Commodity category'] != 'All technologies'].groupby('Commodity category')['Amount (USD)'].sum().reset_index().sort_values('Amount (USD)',
                                                                                        ascending=False))


        bar_1= px.bar( category_totals, x='Commodity category',y='Amount (USD)',
            title='Exports by Commodity Category',color_discrete_sequence=['skyblue'])
        
        bar_1.show()


    with tab3:
        region_totals = (df[df['Region']!='Unknown'].groupby('Region')['Amount (USD)'].sum().reset_index().sort_values('Amount (USD)', ascending=False))


        bar_2= px.bar(region_totals,x='Region', y='Amount (USD)',
            title='Exports by Region',color_discrete_sequence=['deepskyblue'])
        
        bar_2.show()


    with tab4:
        
        mean = df.groupby('Year')['Amount (USD)'].mean().reset_index()
        line = px.line(mean,y='Amount (USD)',x='Year',title='Exports Trend Over Time',color_discrete_sequence=['dodgerblue'])
        line.show()


    with tab5:

        col7,col8,col9=st.columns(3,gap='large')
        col10,col11,col12=st.columns(3,gap='large')
        

        with col7:

            df_battery = df[df['Commodity category']=='Batteries']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_2 = px.line( monthly_avg, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - Batteries')
        
            line_2.show() 


        with col8:

            df_ev = df[df['Commodity category']=='EVs']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_3 = px.line( df_ev, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - EVs')
        
            line_3.show() 

        with col9:
            df_grid = df[df['Commodity category']=='Grid']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_4= px.line( df_grid, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - Grid')
        
            line_4.show() 


        with col10:
            
            df_heat = df[df['Commodity category']=='Heating and cooling systems']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_5= px.line( df_heat, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - Heating and cooling systems')
        
            line_5.show() 



        with col11:
            
            df_solar = df[df['Commodity category']=='Solar PV']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_6= px.line( df_solar, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - Solar PV')
        
            line_6.show() 



        
        with col12:
            
            df_wind = df[df['Commodity category']=='Wind']].groupby('Month')['Amount (USD)'].sum().reset_index()
        
            line_7= px.line( df_wind, x='Month',color_discrete_sequence=['deeppink'],
                y='Amount (USD)', markers=True, title='Monthly Seasonality - Wind')
        
            line_7.show() 

  



# prediction page

else:
    st.image("https://unsplash.com/photos/blue-solar-panel-boards-V4ZYJZJ3W4M")
    

  
    

    # Filters

    st.sidebar.header('Filters')

    country = st.sidebar.selectbox('country',df['Area'].unique())
    filtered_region = df[df['Area']==country]['Region'].unique()
    region = st.sidebar.selectbox('Region',filtered_region)

    category=st.sidebar.selectbox('Commodity category',df['Commodity category'].unique())

    amount = st.sidebar.slider('Amount (USD)',min_value=float(df['Amount (USD)'].min()),
                                                    max_value= float(df['Amount (USD)'].max()))


    year = st.number_input("Year",min_value=2018,max_value=2030,value=2027,step=1)


    month = st.selectbox('Month', list(range(1, 13)))




    
    # Prediction

    if st.button("Rolling Sum Forecasting"):
        df_prediction = pd.DataFrame(data= [[country,region,category,amount,year,month]],
                    columns=df.drop( 'Amount 12 month rolling sum (USD)',axis=1).columns)
        try:
            st.table(df_prediction)
            result = knn.predict(df_prediction)[0]
            prediction = knn.predict(df_prediction)
            st.success(f" Estimated Rolling Sum ': {prediction[0]:,.2f} USD")
        except Exception as e:
            st.error(f"Prediction Error: {e}")




Overwriting /kaggle/working/china_tech.py


In [12]:
! streamlit run china_tech.py



2026-11-04 03:49:19.440 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.158.12.30:8501

^C
  Stopping...
